### RAG Pipeline - Data Ingestion to vector DB pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
### Read all the PDF files inside the directory

def process_all_pdfs(pdf_directory):
    #Process all PDF files in the specified directory and return a list of documents.
    pdf_dir = Path(pdf_directory)
    all_documents = []

    #Find All pdf files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add metadata to each document
            for doc in documents:
                doc.metadata["file_name"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"\nLoaded {len(documents)} pages from {pdf_file}.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Total documents loaded: {len(all_documents)} pages.") 
    return all_documents

#process all pdf files in the data directory
all_pdf_documents = process_all_pdfs("../data")

In [ ]:
all_pdf_documents

In [ ]:
### Text splitting and get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] #Chunking separators in order of preference
        )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #show example of the first chunk
    print("\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...") #print first 200 characters of the first chunk 
    print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks= split_documents(all_pdf_documents)

### Embedding and vector space db

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer ## Embedding model
import uuid ## to generate unique ids for each document chunk in vector database
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import chromadb ## Vector database
from chromadb.config import Settings

In [28]:
class embedding_manager:
    #Handels document embedding using sentence transformers and stores them in chromadb vector database

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize the embedding manager with a specified sentence transformer model."""

        """Args:
            model_name (str): huggingface model name for sentence transformer. Default is 'all-MiniLM-L6-v2'.
        """
        self.model = None
        self.model_name = model_name
        self._load_model() # _ means protected method, can be called inside this class but not outside
         

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of document chunks.
        Args:
            texts: List of text strings to embed.

        Returns:
            A numpy array of shape (num_texts, embedding_dim) containing the generated embeddings.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} document chunks...")
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embedding.shape}")
        return embedding

## Initialize the embedding manager
embedding_manager_instance = embedding_manager()
embedding_manager_instance

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5171.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [29]:
class VectorStore:
    # Manages document embedding in chromadb vector database

    def __init__(
        self,
        collection_name: str = "pdf_chunks",
        persist_directory: str = "../data/vector_store",
    ):
        """Initialize the vector store with a specified collection name and persistence directory.

        Args:
            collection_name (str): Name of the chromadb collection to store document chunks. Default is "pdf_chunks".
            persist_directory (str): Directory path for chromadb persistence. Default is "../data/vector_store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()  # protected method to initialize the vector store

    def _initialize_vector_store(self):
        """Initialize the chromadb client and collection."""
        try:
            print(
                f"Initializing ChromaDB client with persistence directory: {self.persist_directory}..."
            )

            # Create persistence chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            # self.client = chromadb.Client(Settings(chroma_db_impl="duckdb+parquet", persist_directory=self.persist_directory))
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection for document chunks
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Document embedding for RAG system"},
            )
            print(
                f"Vector store initialized successfully with collection: {self.collection_name}"
            )
            print(f"Current number of documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_documents(self, documents: List[Any], embedding: np.ndarray):
        """Add document and their embeddings to the vector store.

        Args:
            documents: List of document chunks to add to the vector store. Each chunk
                should be an object (e.g. a LangChain Document) that has
                `page_content` and `metadata` attributes.
            embedding: NumPy array of embeddings for the documents.
        """
        if len(documents) != len(embedding):
            raise ValueError("The number of documents and embeddings must be the same.")
        print(f"Adding {len(documents)} documents to the vector store...")

        # prepare the data for chromadb
        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []

        for i, (doc, emb) in enumerate(zip(documents, embedding)):
            # Generate a unique ID for each document chunk
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # unique id with prefix doc_ and index
            ids.append(doc_id)

            # Prepare metadata for each document chunk
            metadata = dict(doc.metadata)  # copy the original metadata from the document chunk
            metadata["doc_index"] = i  # add the index of the document chunk in the original documents list
            metadata["content_length"] = len(doc.page_content)  # add the length of the document chunk content
            metadatas.append(metadata)

            # Document content
            document_texts.append(doc.page_content)

            # Embedding
            embedding_list.append(emb.tolist())  # convert numpy array to list for chromadb

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embedding_list,
            )
            print(f"Added {len(documents)} documents to the vector store successfully.")
            print(f"Current number of documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e
        
VectorStore = VectorStore()
VectorStore
    
    
    



Initializing ChromaDB client with persistence directory: ../data/vector_store...
Vector store initialized successfully with collection: pdf_chunks
Current number of documents in the collection: 231


In [30]:
chunks

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-01T08:20:45+00:00', 'moddate': '2025-04-01T15:21:02+05:30', 'source': '..\\data\\pdf\\NCERT CH101.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'file_name': 'NCERT CH101.pdf', 'file_type': 'pdf'}, page_content='Two little hands \ngo clap, clap, clap.\nTwo little legs \ngo tap, tap, tap.\nTwo little eyes \nare open wide.\nOne little head \ngoes side to side.\nUnit 1 \nMy Family and Me\nChapter 1\nTwo Little Hands\n Let us sing\nChapter 1.indd   1 11-01-2024   04:14:39\nReprint 2025-26'),
 Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-01T08:20:45+00:00', 'moddate': '2025-04-01T15:21:02+05:30', 'source': '..\\data\\pdf\\NCERT CH101.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'file_name': 'NCER

In [31]:
### convert the text to embedding and store in vector database
texts = [doc.page_content for doc in chunks]

## Generate embeddings for the document chunks
embeddings = embedding_manager_instance.generate_embeddings(texts)

## Store the document chunks and their embeddings in the vector store
VectorStore.add_documents(chunks, embeddings)

Generating embeddings for 71 document chunks...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.33it/s]


Generated embeddings with shape: (71, 384)
Adding 71 documents to the vector store...
Added 71 documents to the vector store successfully.
Current number of documents in the collection: 302


### Retriver pipeline from vectore store

In [32]:
class RagRetriever:
    #Retrieves relevant document chunks from the vector store based on a query

    def __init__(self, vector_store: VectorStore, embedding_manager: embedding_manager):
        """Initialize the RAG retriever with a vector store, embedding manager, and number of top results to retrieve.

        Args:
            vector_store (VectorStore): An instance of the VectorStore class to retrieve documents from.
            embedding_manager (embedding_manager): An instance of the embedding_manager class to generate query embeddings.
            top_k (int): The number of top relevant document chunks to retrieve. Default is 5.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant document chunks from the vector store based on a query.

        Args:
            query (str): The input query string
            top_k (int): The number of top relevant document chunks. Default is 5.
            score_threshold (float): Minimum cosine similarity score for retrieved documents. Default is 0.0."""
        
        print(f"Retrieving relevant documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}...")
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # (Search) Retrieve relevant document chunks from the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                #include=["documents", "metadatas", "embeddings"],
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, documnet, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distance to similarity score (assuming distance is cosine distance, similarity = 1 - distance) chromadb use cosine distance by default
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": documnet,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,  # Rank starts from 1
                        })

                print(f"Retrieved {len(retrieved_docs)} relevant documents from the vector store.")
            else:
                print("No relevant documents found for the query.")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error retrieving documents from the vector store: {e}")
            return []
        
rag_retriver = RagRetriever(VectorStore, embedding_manager_instance)


In [33]:
rag_retriver.retrieve("What Five Little Monkeys are doing.")


Retrieving relevant documents for query: 'What Five Little Monkeys are doing.' with top_k=5 and score_threshold=0.0...
Generating embeddings for 1 document chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.73it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 relevant documents from the vector store.


[{'id': 'doc_c8fd9add_49',
  'content': '50\nMridang\nFive little monkeys \njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFour little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFive Little Monkeys\nThree little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nLet us sing\nChapter 2.indd   50 12-01-2024   04:37:57\nReprint 2025-26',
  'metadata': {'total_pages': 7,
   'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'file_type': 'pdf',
   'doc_index': 49,
   'creationdate': '2025-04-01T08:23:27+00:00',
   'moddate': '2025-04-01T15:21:04+05:30',
   'page_label': '4',
   'file_name': 'NCERT CH103.pdf',
   'page': 3,
   'source': '..\\data\\pdf\\NCERT CH103.pdf',
   'content_length': 360},
  'similarity_score': 0.16297900676727295,
  'distance': 0.837020993232727,
  'rank': 1},
 {'id': 'doc_de107674_49',
  'content': '50\nMrid

### Integration VectorDB Context Pipeline With LLM Output

In [34]:
# simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize Groq LLM (Set your Groq API key in the .env file as GROQ_API_KEY)
#groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key = ""

llm = ChatGroq(
    groq_api_key=groq_api_key, 
    model="groq/compound",      # 'model' is preferred over 'model_name'
    temperature=0.1,           # Added the missing comma here!
    max_tokens=1024
)

# Simple Rag Function : retrive context +  genearte response.
def rag_simple(query, retriever, llm, top_k=3):
    
    ## Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        print("No relevant context found for the query.")

    ## Generate response using Groq LLM
    prompt=f""" Use the following retrieved context to answer the question concisely.
        context: {context}
        Question: {query}
        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response    

In [35]:
answer = rag_simple("What are Five Little Monkeys doing?", rag_retriver, llm)
print(answer.content)


Retrieving relevant documents for query: 'What are Five Little Monkeys doing?' with top_k=3 and score_threshold=0.0...
Generating embeddings for 1 document chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 relevant documents from the vector store.


They are jumping on a tree.


In [36]:
### Enhance Rag pipeline with amazing features

from pydoc import doc

def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    Rag pipeline with advanced features:
        return answer, sources, confidence score, and optionally the retrieved context.
    """
    results = retriever.retrieve(query, top_k=top_k,score_threshold=min_score)
    if not results:
        return {"answer": "No relevant information found.", "sources": [], "confidence": 0.0, "context": []}
    
    # prepare context and source
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [{
        'source':doc['metadata'].get('file_name', 'unknown') + " (page: " + str(doc['metadata'].get('page', 'unknown')) + ")",
        'page':doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview_score': doc['content'][:300] + '...'  # first 300 chars as preview
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)  # confidence score based on the top retrieved document
 
    # Generate answer using Groq LLM
    prompt = f""" Use the following retrieved context to answer the question concisely.\nContext: {context}\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence,
        }
    if return_context:
        output["context"] = context
    return output

#Example usage of the advanced RAG pipeline
result = rag_advance("What are Five Little Monkeys doing?", rag_retriver, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence Score:", result["confidence"])
print("Context preview:", result["context"][:300])

Retrieving relevant documents for query: 'What are Five Little Monkeys doing?' with top_k=3 and score_threshold=0.1...
Generating embeddings for 1 document chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 relevant documents from the vector store.


Answer: The five little monkeys are **jumping on a tree**.
Sources: [{'source': 'NCERT CH103.pdf (page: 3)', 'page': 3, 'score': 0.17819744348526, 'preview_score': '50\nMridang\nFive little monkeys \njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFour little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFive Little Monkeys\nThree little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nLet us si...'}, {'source': 'NCERT CH103.pdf (page: 3)', 'page': 3, 'score': 0.17819744348526, 'preview_score': '50\nMridang\nFive little monkeys \njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFour little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nFive Little Monkeys\nThree little monkeys\njumping on a tree,\nOne fell down\nand bumped his knee.\nAh! Ah! Ah!\nLet us si...'}, {'source': 'NCERT CH103.pdf (page: 3)', 'page': 3, 'score': 0.17819744348526, 'preview_sc